## MCP Servers with Foundry Agents

### Installing Required Libraries

In [1]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Setting Up the Environment Variables

In [2]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")

### Setting up the Foundry Project Client

In [3]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Creating the OpenAI Client

In [4]:
openai_client = project_client.get_openai_client()

### Getting the MCP Server Connection ID

In [5]:
connection_id = ""

for connection in project_client.connections.list():
    if connection.name == mcp_server_name:
        connection_id = connection.id
        break

print(f"The MCP Server Connection ID is: {connection_id}")

The MCP Server Connection ID is: /subscriptions/8bb477c6-092b-4864-b0fd-d10d75f245a9/resourceGroups/azurestalwart/providers/Microsoft.CognitiveServices/accounts/tavant-mf/projects/proj-default/connections/mslearn-mcp-server


### Creating the MCP Tool Spec

In [6]:
tool = MCPTool(
    server_label = "microsoft_learn_server",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="never",
    project_connection_id=connection_id
)

### Creating the MCP Agent

In [7]:
agent = project_client.agents.create_version(
    agent_name="MCP-Agent",
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are an intelligent assistant that can interact with the Microsoft Learn MCP server to provide users with relevant learning resources and information about Microsoft technologies.",
        tools=[tool],
    )
)

print(f"Created MCP Agent with ID: {agent.id}")

Created MCP Agent with ID: MCP-Agent:1


### Creating a Conversation Object for the Agent Chat System

In [8]:
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

Created conversation with id: conv_b65393f285b1013e00bHVzTkBI9cQ6iYVz8qNVVZNeiWQZSzSF


### Chat with the Agent

In [ ]:
user_query = "Can you pls help me with the latest Microsoft Foundry SDK code samples?" 

# Can also try this query: "Find me learning paths on Azure AI services for building intelligent applications."

In [10]:
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={
        "agent": {
            "name": "MCP-Agent",
            "type": "agent_reference"
        }
    },
    input=user_query
)

print(f"Agent Response: {response.output_text}")

Agent Response: Here are some code samples for working with the AI Foundry SDK:

### Integrate FoundryLocalManager with OpenAI Python SDK

```python
import openai
from foundry_local import FoundryLocalManager

# Using an alias to download the most suitable model to your device.
alias = "qwen2.5-0.5b"

# Create a FoundryLocalManager instance. This starts the Foundry Local service and loads the model.
manager = FoundryLocalManager(alias)

# Configure the OpenAI client to use the local Foundry service
client = openai.OpenAI(
    base_url=manager.endpoint,
    api_key=manager.api_key  # API key not required for local usage
)

# Set the model and generate a streaming response
stream = client.chat.completions.create(
    model=manager.get_model_info(alias).id,
    messages=[{"role": "user", "content": "What is the golden ratio?"}],
    stream=True
)

# Print the streaming response
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.con